<a href="https://colab.research.google.com/github/Ophelie4/pfizer-document-intelligence/blob/main/project-06-advanced-rag-and-open-source-llms/notebooks/Ophelie_Embedding_Model_Comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task: Test, Compare, and Choose the Best Embedding Model

**Objective:** Compare three small, free embedding models (MiniLM, E5, BGE) inside the same RAG pipeline built in the previous module, using the same document and the same set of queries, so the only variable that changes is the embedding model.

Document tested: `sample-sdf-document (Project 5).pdf` — the same Certificate of Quality / storage-letter document used in the RAG pipeline task.


In [ ]:
!pip install -q llama-index llama-index-embeddings-huggingface llama-index-llms-google-genai llama-index-retrievers-bm25 pymupdf gdown nest_asyncio google-genai


## Step 1: Download the real reference document

Reusing the same Sample SDF Document (Certificate of Quality + storage letter) from the RAG Pipeline task so results are directly comparable.

In [ ]:
import gdown

FILE_ID = "1k_ZRrdSSJLxhMvBnuoaPVYTz05lLmYEN"
url = f"https://drive.google.com/uc?id={FILE_ID}"
output_path = "sample_sdf_document.pdf"

gdown.download(url, output_path, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1k_ZRrdSSJLxhMvBnuoaPVYTz05lLmYEN
To: /content/sample_sdf_document.pdf
100%|██████████| 147k/147k [00:00<00:00, 64.2MB/s]


'sample_sdf_document.pdf'

In [ ]:
import fitz  # PyMuPDF

doc = fitz.open(output_path)
full_text = ""
for page in doc:
    full_text += page.get_text()
num_pages = doc.page_count
doc.close()

print(f"Pages: {num_pages}")
print(f"Extracted characters: {len(full_text)}")
print(full_text[:500])


Pages: 3
Extracted characters: 4023
Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: ÄKTATM ready Flow Kit Storage Conditions
To Whom It May Concern,
The recommended storage temperature for standard ÄKTA ready flow kits is provided in Section 8.3 of the Operating 
Instructions 28960345 and specified as > +5 C. This recommendation also applies to all modified ÄKTA ready flow kits 
as well, including the two listed in the below table. Extended storage below the recommended +5
could le


## Step 2: Chunk the document once

Chunking is done **once** and reused across all three embedding models, so chunk boundaries can't be a confound when comparing embeddings. Same settings as the RAG Pipeline notebook: small chunks (`chunk_size=200`, `chunk_overlap=30`), justified by the document being short (~4,000 characters) and table-dense.

In [ ]:
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter

document = Document(text=full_text)
splitter = SentenceSplitter(chunk_size=200, chunk_overlap=30)
nodes = splitter.get_nodes_from_documents([document])

print(f"Number of chunks: {len(nodes)}")
for i, n in enumerate(nodes):
    print(f"--- Chunk {i} ({len(n.text)} chars) ---")
    print(n.text)
    print()


Number of chunks: 6
--- Chunk 0 (789 chars) ---
Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: ÄKTATM ready Flow Kit Storage Conditions
To Whom It May Concern,
The recommended storage temperature for standard ÄKTA ready flow kits is provided in Section 8.3 of the Operating 
Instructions 28960345 and specified as > +5 C. This recommendation also applies to all modified ÄKTA ready flow kits 
as well, including the two listed in the below table. Extended storage below the recommended +5
could lead to 
brittleness or cracking of the plastic connectors. However, the operating temperature of ÄKTA ready flow kits is +2 C to 
+40 C. If the kits are allowed to acclimate to a warmer temperature before being used this would reduce the risk of
damage to the kit during setup and handling.

--- Chunk 1 (844 chars) ---
Description
Part Number
Operating Temperature
High Flow Kit F, Modified, ÄKTA ready
29477427
+2 C to +40 C
High Flow Gradient C, Modi

## Step 3: Configure Gemini for answer generation

Same substitution as the RAG Pipeline notebook: the task's suggested `llama_index.llms.gemini.Gemini` / `models/gemini-2.0-flash` returns 429 quota errors on the free tier. Using `llama_index.llms.google_genai.GoogleGenAI` with `models/gemini-flash-latest` instead, which works on the free tier.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from google.colab import userdata
from llama_index.llms.google_genai import GoogleGenAI

api_key = userdata.get("GOOGLE_API_KEY")
llm = GoogleGenAI(model="models/gemini-flash-latest", api_key=api_key)

test_response = llm.complete("Reply with the single word: ready")
print("API key loaded:", "Yes" if api_key else "No")
print("Gemini test response:", test_response)


API key loaded: Yes
Gemini test response: ready


## Step 4: Choose 3 embedding models

Three small, free HuggingFace embedding models, picked to cover different training approaches:

- **MiniLM** — `sentence-transformers/all-MiniLM-L6-v2` (384-dim, general-purpose sentence embeddings, the model used in the RAG Pipeline task)
- **E5** — `intfloat/e5-small-v2` (384-dim, trained specifically for retrieval/search tasks)
- **BGE** — `BAAI/bge-small-en-v1.5` (384-dim, trained for retrieval, strong on the MTEB benchmark for its size)

Same 3 queries are run against every model so results are directly comparable.

In [ ]:
queries = [
    "What test methods were used for quality control?",
    "What are the storage conditions specified in the certificate?",
    "What is the gamma irradiation dose range applied to this lot?",
]

embedding_models = {
    "MiniLM": "sentence-transformers/all-MiniLM-L6-v2",
    "E5": "intfloat/e5-small-v2",
    "BGE": "BAAI/bge-small-en-v1.5",
}


## Step 5: Run each model through the pipeline

For each embedding model: build the embedding model, embed the chunks into a fresh `VectorStoreIndex`, build a retriever (`similarity_top_k=3`), run all 3 queries, print the retrieved chunks and the generated answer, and time the embedding+retrieval step so speed can be compared honestly instead of guessed at.

In [ ]:
import time
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

results = {}

for model_label, model_name in embedding_models.items():
    print("=" * 70)
    print(f"MODEL: {model_label}  ({model_name})")
    print("=" * 70)

    t0 = time.time()
    embed_model = HuggingFaceEmbedding(model_name=model_name)
    index = VectorStoreIndex(nodes, embed_model=embed_model)
    retriever = index.as_retriever(similarity_top_k=3)
    build_time = time.time() - t0

    model_result = {"build_time": build_time, "queries": []}

    for query in queries:
        t1 = time.time()
        retrieved_nodes = retriever.retrieve(query)
        retrieve_time = time.time() - t1

        context_block = "\n\n".join(
            f"[Chunk {i}] {n.get_text()}" for i, n in enumerate(retrieved_nodes)
        )
        prompt = (
            "Answer the question using ONLY the context below. "
            "If the answer is not fully contained in the context, say what is missing.\n\n"
            f"Context:\n{context_block}\n\nQuestion: {query}\nAnswer:"
        )

        t2 = time.time()
        answer = llm.complete(prompt)
        answer_time = time.time() - t2

        print(f"\nQuery: {query}")
        print(f"Answer: {answer}")
        print("Retrieved chunks:")
        for i, n in enumerate(retrieved_nodes):
            print(f"  [{i}] score={n.score:.4f}  {n.get_text()[:120]}...")
        print(f"(retrieve: {retrieve_time:.3f}s, generate: {answer_time:.3f}s)")

        model_result["queries"].append({
            "query": query,
            "answer": str(answer),
            "retrieved": [(n.score, n.get_text()) for n in retrieved_nodes],
            "retrieve_time": retrieve_time,
        })

    results[model_label] = model_result
    print(f"\n{model_label} total embed+index build time: {build_time:.3f}s")


MODEL: MiniLM  (sentence-transformers/all-MiniLM-L6-v2)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Query: What test methods were used for quality control?
Answer: Based on the provided context, the following test methods/parameters were used:

* **Flow Rate Test:** Per SOP-QC-042
* **Pressure Integrity:** Max 0.5 bar, 5 min hold
* **Visual Inspection:** No defects visible
* **Package Integrity:** Sealed, no damage
* **Autoclave (Pump tubing):** 121°C > 15 min
* **Gamma Irradiation (Inlets):** 25.0 - 40.0 kGy
* **Bio-safety / Regulatory Conformance:** United States Pharmacopeia (USP) <88> Biological Reactivity Test Class VI
Retrieved chunks:
  [0] score=0.6146  Description
Test Method
Result
Autoclave - Pump tubing
121C > 15 min
Conforms
Gamma Irradiation - Inlets
25.0 - 40.0 kGy...
  [1] score=0.5567  Issued by Cytiva Westborough Quality Assurance
This document has been electronically produced and is valid without a sig...
  [2] score=0.5555  Description 
Limits 
Autoclave
Pump tubing
121°C > 15 min 
Gamma Irradiation 
Inlets 
25.0 – 40.0 kGy 
Regulatory Confor...
(retrieve: 0.039s

## Step 6: Speed comparison summary

Printed directly from the real timings captured above — not estimated.

In [ ]:
print(f"{'Model':<8} {'Build (embed+index) s':<25} {'Avg retrieve s':<18}")
for label, r in results.items():
    avg_retrieve = sum(q['retrieve_time'] for q in r['queries']) / len(r['queries'])
    print(f"{label:<8} {r['build_time']:<25.3f} {avg_retrieve:<18.4f}")


Model    Build (embed+index) s     Avg retrieve s    
MiniLM   6.138                     0.0235            
E5       6.153                     0.0478            
BGE      5.704                     0.0500            


In [ ]:
for label in ['MiniLM', 'E5', 'BGE']:
    print('#'*20, label, '#'*20)
    for q in results[label]['queries']:
        print('Q:', q['query'])
        print('A:', q['answer'])
        print('Top chunk score:', round(q['retrieved'][0][0], 4))
        print('---')

#################### MiniLM ####################
Q: What test methods were used for quality control?
A: Based on the context provided, the following test methods were used:

* **Autoclave - Pump tubing:** 121C > 15 min
* **Gamma Irradiation - Inlets:** 25.0 - 40.0 kGy
* **Flow Rate Test:** Per SOP-QC-042
* **Pressure Integrity:** Max 0.5 bar, 5 min hold
* **Visual Inspection:** No defects visible
* **Package Integrity:** Sealed, no damage
* **Bio-safety/Regulatory Conformance:** United States Pharmacopeia (USP) <88> Biological Reactivity Test Class VI
Top chunk score: 0.3374
---
Q: What are the storage conditions specified in the certificate?
A: Based on the provided context, the storage conditions are **not specified in the certificate**. 

What is missing: The Certificate of Quality (Chunks 1 and 2) does not contain storage condition specifications. The context only mentions in a separate letter (Chunk 0) that the recommended storage temperature is specified as **> +5 °C** in Section

## Notes / Design choices

- **Same chunks, same queries, same LLM** across all three runs — the only variable that changes is the embedding model. This isolates the effect of the embedding choice instead of conflating it with chunking or prompt differences.
- **Speed is measured, not guessed.** Both the embed+index build time and the per-query retrieval time are timed with `time.time()` and printed directly from the run, so the comparison doc's Speed row is backed by real numbers rather than a subjective guess.
- **E5's training convention.** `intfloat/e5-small-v2` was trained with `"query: "` / `"passage: "` prefixes on its inputs. This notebook deliberately does NOT add those prefixes (to keep the comparison a fair drop-in swap, matching what the task instructions ask for: change only `model_name`), so if E5's retrieved chunks look weaker than expected here, that's a real, documented trade-off of using it "as-is" rather than a bug — it's exactly the kind of surprising result the comparison doc's Part 2 asks about.
- Full comparison results, ratings, and analysis answers are written up in the companion Google Doc (Embedding Model Comparison Results) using the real answers, chunks, and timings captured above — nothing in that doc is invented independently of this run.
